# Imports

In [ ]:
# my functions
import helper_functions as hf

# data handling 
import pandas as pd 
import geopandas as gpd
import numpy as np

# plotting 
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import seaborn as sns 

# mesa stuff
import mesa
from mesa import Agent, Model
from mesa.space import ContinuousSpace
from mesa.visualization import SolaraViz, make_plot_component, make_space_component
from mesa.visualization.utils import update_counter
from mesa.datacollection import DataCollector
from mesa.batchrunner import batch_run
#import solara

# rando stuff
import time
from scipy.stats import truncnorm
import random

# set notebook options
pd.set_option('display.max_rows', 1000)


# Load Road and Modify

In [ ]:
#road_gdf = gpd.read_file("data/roads/hw210_w_speed_limits.geojson")
road_gdf = gpd.read_parquet("data/roads/hw210_w_speed_limits.parquet")

print(f'number of road points: {len(road_gdf)}')
road_gdf.plot()
print('')
#print('GDF Head', f'GDF CRS:{road_gdf.crs} ')
display(road_gdf.head(3))

# Adjust Speed Helper Functions

In [ ]:

def get_acceleration(current_speed, max_speed=hf.get_dps(60), max_accel=hf.get_dps(4)):
    """
    Estimate acceleration in degrees per minute based on current speed.

    Parameters:
        current_speed (float): Current speed in degrees per secound
        max_speed (float): Max speed in degrees per secound
        max_accel (float): Peak acceleration in mph/s defaults to a change in 4 mph/sec.

    Returns:
        float: Acceleration in degrees per minute²
    """
    # Normalize current speed as a fraction of max speed
    x = current_speed / max_speed
    x = np.clip(x, 0, 1)
    # Logistic-shaped acceleration curve (derivative of sigmoid)
    accel = max_accel * (1 - x) * x * 4  # Scaling factor 4 centers the peak at 0.5
    return accel

def get_deceleration(how='soft'):
    options = {'soft': hf.get_dps(1), #~1mph/s 
               'normal': hf.get_dps(2.5), #~2.5mph/s
               'hard': hf.get_dps(5) #~5mph/s
              }
    return  options[how]
    



# RoadSegmentAgent

In [ ]:
class RoadSegmentAgent(mesa.Agent):
    """Represents a segment of the road. Only one car can occupy it at a time."""
    
    def __init__(self, model, position, speed_limit, linked_coord):
        super().__init__(model)
        self.position = position  # The index of the segment
        self.occupied = False  # Whether a car is on this segment
        self.status = 'im just a road'
        self.speed_limit = speed_limit
        self.linked_coord = linked_coord

    def step(self):
        """Tracks occupancy but does not move."""
        pass

# VehicleAgent base class

In [ ]:
class VehicleAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.status = "driving"
        self.speed = hf.get_dps(10) # starting speed
        
        # For data collection 
        self.created_at_step = self.model.steps 
        self.steps_taken = 0 
        self.distance_traveled = 0
        self.car_interactions = 0
        self.gap = 0 
        self.next_agent = None
        self.driving_action = None

        # These will be set in subclasses
        self.max_speed = None
        self.ideal_distance_multiplier = None
        self.acceptable_over = None

        # Speed control tuning parameters (can be overridden)
        self.jitter_threshold = hf.get_dps(2)
        self.jitter_variance = hf.get_dps(0.2)
        self.max_acceleration = hf.get_dps(4)
        self.min_speed = hf.get_dps(1)
        self.deceleration_mode = 'normal'

        # init all the road segement data 
        self.road_segments = self.model.agents.select(agent_type=RoadSegmentAgent)

        # Establish the Vehicles positio n 
        self.path = self.road_segments.get('position')
        self.path_index = 0
        self.model.space.place_agent(self, self.path[0])  # <-- here is the intial place agent

        
    def end_of_road(self):
        '''If at the last segment, remove the car'''
        if self.path_index >= len(self.path) - 1:
            self.status = "arrived"
            
            self.model.finished_agents.append({
                "AgentID": self.unique_id,
                "created_at_step": self.created_at_step,
                "steps_taken": self.steps_taken,
                "car_interactions": self.car_interactions, 
                "distance_traveled": self.distance_traveled*69 , # times 69 to convert ~miles
                "approx_average_mph": (self.distance_traveled*69)/(self.steps_taken/3600), 
                "acceptable_over": hf.get_mph(self.acceptable_over),
                "ideal_distance_multiplier":self.ideal_distance_multiplier
            # Add more if needed
            })
            self.remove() 
            return True


    def get_next_agent(self): 
        '''
        Used in the get_gap function
        Takes self, checks if a next_agent exists and status == driving, if so uses that, if not trys to find a new next agent. 
        '''
        # check if 1) next car is already saved & 2)it is driving. This works because self.next_agent existing is tested first
        if self.next_agent and self.next_agent.status == "driving":
             return
        
        # if the next agent does not exist look for a new next_agent
        other_vehicles = self.model.agents.select(agent_type=VehicleAgent)
        cars_ahead = [
            agent for agent in other_vehicles
            if agent.distance_traveled > self.distance_traveled
        ]
        
        # set the next agent to the next vehicle, if no next vehicle then set to None
        if cars_ahead:
            self.next_agent = min(cars_ahead, key=lambda agent: agent.distance_traveled)
        else:
            self.next_agent = None
            
        
    def get_gap(self):
            """
            Returns:
                ideal_gap: float — the desired following distance (deg)
                gap: float — distance to the closest vehicle ahead (deg)

            Used in the adjust_speed function
            """
            ideal_gap = self.speed * self.ideal_distance_multiplier  
            # run the get_next_agent function 
            self.get_next_agent()
            
            # if a agent exists then measure the gap or set to inf 
            if self.next_agent: 
                gap = model.space.get_distance(self.pos, self.next_agent.pos)
            else:
                gap = float("inf")

            self.gap = gap
            return ideal_gap, gap
        
    def get_speed_limit(self):
        return hf.get_dps(self.road_segments[self.path_index].speed_limit) + self.acceptable_over

    def smooth_brake(self, gap, ideal_gap):
        """
        Calculate deceleration based on a smooth braking rule,
        where speed and distance are in degrees per second and degrees.
        """
        if ideal_gap <= 0 or np.isnan(ideal_gap):
            force = 0
        else:
            force = max((ideal_gap - gap)/ideal_gap, 0) # this produces a number between 0 and 1
            
        deceleration = force * hf.get_dps(5) # <- this is acting as max decel 
        return deceleration
            
    def adjust_speed(self):
        ''' takes self from self uses'''
        
        ideal_gap, gap  = self.get_gap() 
        speed_limit = self.get_speed_limit()
        
        
        # 1) measues the gap to the next vehicle, if less than the ideal gap, applies the smooth breaking
        if gap < ideal_gap:
            self.driving_action = 'smooth_break'
            self.car_interactions += 1
            self.speed -= max(self.smooth_brake(gap=gap, ideal_gap=ideal_gap), get_deceleration(how=self.deceleration_mode))
        # 2)if the gap is > than the ideal gap see if the car vehicle is around the speed limit, if so adjust by the jitter
        elif abs(self.speed - speed_limit) < self.jitter_threshold:
            self.driving_action = 'jitter'
            self.speed += self.random.normalvariate(0, self.jitter_variance)
        # 3) if outside the jitter threashhold see if the car is above speed limit, if so break
        elif self.speed > speed_limit:
            self.driving_action = 'speed_limit_break'
            self.speed = max(self.speed - get_deceleration(how=self.deceleration_mode), self.min_speed)
        # 4) if outside the jitter threashhold & below speed limit & max speed then speed up 
        elif self.speed < self.max_speed:
            self.driving_action = 'accelerate'
            self.speed += min(get_acceleration(self.speed, self.max_speed, self.max_acceleration), speed_limit - self.speed)

        # overwrites
        if self.speed > gap: 
            self.driving_action = 'prevent_pass'
            self.speed = gap
        # else:
            
        
        # # stop gap so the car can never pass the car infront if it
        # self.speed = min(self.speed, gap)

    def move_along_path(self):
        """Move the agent along its predefined path based on current speed."""
        distance_to_travel = self.speed
        self.distance_traveled += distance_to_travel
        pos = np.array(self.pos)
        new_position = pos
        
        while distance_to_travel > 0 and not self.end_of_road():
            next_target = np.array(self.path[self.path_index + 1])
            direction = model.space.get_heading(pos, next_target)
            distance = model.space.get_distance(pos, next_target)
    
            if distance < distance_to_travel:
                self.path_index += 1
                distance_to_travel -= distance
                pos = next_target
                new_position = pos
            else:
                step_vector = distance_to_travel * direction / distance
                new_position = pos + step_vector
                distance_to_travel = 0
    
        self.model.space.move_agent(self, tuple(new_position))

    def step(self):
        
        if self.status == "arrived":
            return 
        
        self.steps_taken += 1  
        self.adjust_speed()
        self.move_along_path()


# Spicific VehicleAgent Classes

In [ ]:
class CarAgent(VehicleAgent):
    """Represents a car moving in the canyon."""
    def __init__(self, model, road_points_gdf):
        super().__init__(model)
        self.status = "driving"  # the initial status of the car
        
        # speed perams
        self.max_speed = hf.get_dps(80)
        self.acceptable_over = hf.get_dps(truncnorm((-2 - 3)/4, (20 - 3)/4, loc=3, scale=4).rvs()) # this is a right skewed normal dist bounded by (-2,20)
        self.ideal_distance_multiplier = truncnorm((1.2 - 1.5)/.2, (2.5 - 1.5)/.2, loc=1.5, scale=.2).rvs()
        
        # Speed control tuning parameters 
        self.jitter_threshold = hf.get_dps(3)
        self.jitter_variance = hf.get_dps(0.2)
        self.max_acceleration = hf.get_dps(4)
        self.min_speed = hf.get_dps(1)
        self.deceleration_mode = 'normal'
        
class BusAgent(VehicleAgent):
    """Represents a bus moving in the canyon."""
    def __init__(self, model, road_points_gdf):
        super().__init__(model)
        self.status = "driving"  # the initial status of the car
        
        # speed perams
        self.max_speed = hf.get_dps(60)
        self.acceptable_over = 0
        self.ideal_distance_multiplier = 2.5
        
        # Speed control tuning parameters 
        self.jitter_threshold = hf.get_dps(3)
        self.jitter_variance = hf.get_dps(0.2)
        self.max_acceleration = hf.get_dps(1.5)
        self.min_speed = hf.get_dps(1)
        self.deceleration_mode = 'normal'

    

# Model 

In [ ]:

class TrafficModel(mesa.Model):
    """Mesa model simulating traffic on the canyon road with a car cap."""

    def __init__(self, road_points_gdf=None, p_generate=.001, max_cars=50, max_steps=50000, seed=None, log_agents=False):
        super().__init__(seed=seed)
        self.log_agents = log_agents
        self.p_generate = p_generate  # Probability of new car each step
        self.max_cars = max_cars  # Maximum number of cars allowed
        self.road_points_gdf = road_points_gdf
        self.max_steps = max_steps
        
        # verious trackers
        self.too_close_tracker = 0 
        self.cars_generated = 0 
        self.finished_agents = [] 
            
        # Set up ContinuousSpace
        buffer = .0001
        minx, miny, maxx, maxy = road_points_gdf.total_bounds
        self.space = ContinuousSpace(
            x_min=minx - buffer,
            x_max=maxx + buffer,
            y_min=miny - buffer,
            y_max=maxy + buffer,
            torus=False
        )

        # Create road segment agents - this just creates them in a loop setting the position via the gdf point
        self.road_segments = RoadSegmentAgent.create_agents( 
            model=self, 
            n=len(self.road_points_gdf), 
            position=[(point.x, point.y) for point in self.road_points_gdf.geometry], # need to be passed as a list
            speed_limit=[speed_limit for speed_limit in self.road_points_gdf.speed_limit],
            linked_coord=[linked_coord for linked_coord in self.road_points_gdf.linked_coord]
        )
        # place all the road segments in space - goes hand in hand with read point reation 
        for agent, point in zip(self.road_segments, road_points_gdf.geometry):self.space.place_agent(agent, (point.x, point.y))

        # establish the data collector 
        agent_reporters={
            "AgentType": lambda a: a.__class__.__name__ ,
            'status': lambda a: a.status if isinstance(a, VehicleAgent) else None,
            'driving_action': lambda a: a.driving_action if isinstance(a, VehicleAgent) else None,
            'speed': lambda a: hf.get_mph(a.speed) if isinstance(a, VehicleAgent) else None,
            'steps_taken': lambda a: a.steps_taken if isinstance(a, VehicleAgent) else None,
            'pos':lambda a: a.pos if isinstance(a, VehicleAgent) else None,
            'gap_ft':lambda a: hf.degrees_to_feet(a.gap) if isinstance(a, VehicleAgent) else None,
            "next_vehicle": lambda a: a.next_agent.unique_id if isinstance(a, VehicleAgent) and a.next_agent is not None else None,
            "next_vehicle_status": lambda a: a.next_agent.status if isinstance(a, VehicleAgent) and a.next_agent is not None else None,


            #'hrs': lambda a: a.steps_taken/3600 if isinstance(a, CarAgent) else None,
            #'distance_traveled': lambda a: a.distance_traveled if isinstance(a, CarAgent) else None,
            #'acceptable_over': lambda a: hf.get_mph(a.acceptable_over) if isinstance(a, CarAgent) else None,
            #'ideal_distance_multiplier': lambda a: a.ideal_distance_multiplier if isinstance(a, CarAgent) else None,
        }

        model_reporters={
            "cars_generated": lambda m: m.cars_generated,
            "too_close_tracker": lambda m: m.too_close_tracker, 
            "FinishedAgentsSummary": lambda m: None  # Placeholder
        }

        if log_agents:
            self.datacollector = DataCollector(
                model_reporters = model_reporters, 
                agent_reporters = agent_reporters
            )
        else: 
            self.datacollector = DataCollector(model_reporters = model_reporters)
    
    def generate_new_car(self):
        # Only generate if under max limit
        if self.cars_generated >= self.max_cars:
            return
        # Get the starting point
        start_point = self.road_points_gdf.iloc[0].geometry.coords[0]  
        
        # Check if another car is too close to the start
        too_close = any(
            self.space.get_distance(agent.pos, start_point) < (5/100000) # m -> degrees
            for agent in self.agents.select(agent_type=CarAgent)[-5:]
        )
        if too_close:
            self.too_close_tracker += 1
        elif self.random.random() < self.p_generate:
            CarAgent.create_agents(model=self, n=1, road_points_gdf=self.road_points_gdf)
            self.cars_generated += 1
    
    def model_stop_process(self):
        # add agent summary data to the datacollector
        self.datacollector.model_vars["FinishedAgentsSummary"][-1] = self.finished_agents
        self.running = False
        
    def step(self):
        # Collect data before stepping
        self.datacollector.collect(self)

        # generate a new car based on a simple probability 
        self.generate_new_car()

        # Shuffle agent execution and step them - this calls the step functions of the agents
        #self.agents.shuffle_do("step")
        self.agents.do("step")

        # Stop model when all generated cars have been removed
        if self.cars_generated == self.max_cars:
            remaining_cars = self.agents.select(agent_type=VehicleAgent)
            if len(remaining_cars) == 0:
                print("All cars have been removed. Stopping model.")
                self.model_stop_process()
        
        # Stop model at hard cap of steps
        if self.steps >= self.max_steps:
            print(f"Reached max step count ({self.max_steps}). Stopping model.")
            self.model_stop_process()
             


# Simple model run (fast)

In [ ]:
%%time
model = TrafficModel(road_points_gdf=road_gdf, p_generate=0.001, max_cars=10, max_steps=500000, log_agents=True, seed=1)

while model.running: 
    model.step()

#for i in range(15):
#    model.step()

print(f'Model ran for {model.steps} steps')

## Analyze data

### Finished agents

In [ ]:
# process the finished_agents data 
finished_agents = hf.make_finished_agents_df(model.finished_agents)

print(f'N cars: {len(finished_agents)}, fake hours: {round(model.steps/3600,1)}')

print('Slowest Five Cars')
display(finished_agents.sort_values(by='steps_taken', ascending=False).head())

# Make the nice histogram
hf.make_travel_time_hist(finished_agents)


### Marginal effects model and visulizations 

### All agents - only works when log_agents=True

In [ ]:
# collect the agent data
agent_data = model.datacollector.get_agent_vars_dataframe().reset_index()

# filter for only car agents and manipulate a bit
cars_full = agent_data.loc[agent_data.AgentType == 'CarAgent'].copy()
cars_full['x'] = cars_full['pos'].apply(lambda p: p[0] if isinstance(p, tuple) else p.x)
cars_full['y'] = cars_full['pos'].apply(lambda p: p[1] if isinstance(p, tuple) else p.y)

In [ ]:
# run the animation
hf.animate_traffic(cars_full, road_gdf, interval=60, step_skip=20, watch=None, zoom=1 )


### Issue car analysis

In [ ]:
# looking at one car
issue_car_id = 483
issue_step = 1180

issue_car_df = cars_full.loc[(cars_full.AgentID==issue_car_id)]
print(f'Start of issue: {issue_car_df.loc[issue_car_df.speed<10]["Step"].min()}')
display(sns.scatterplot(data=issue_car_df, x='Step', y='speed'))

issue_car_df.loc[issue_car_df.Step > issue_step].head(10)

In [ ]:
# looking at a group of issue cars
issue_car_ids = [issue_car_id-2,issue_car_id-1, issue_car_id, issue_car_id+1]
issue_car_df = cars_full.loc[cars_full.AgentID.isin(issue_car_ids)]
print(issue_car_ids)
issue_car_df.loc[issue_car_df.Step > issue_step].head(10)

# Model Exe w/ live anamation - not working

In [ ]:
%%time 
# this has to portrayal logic for all the agents
def agent_portrayal(agent):
    if isinstance(agent, CarAgent):
        return {
            "color": "red",
            "size": 15,
        }
    elif isinstance(agent, RoadSegmentAgent):
        return {
            "color": "blue",
            "size": 5,
        }
    return {}


# UI controls – only expose max_cars
model_params = {
    # "max_cars": {
    #     "type": "SliderInt",
    #     "value": 1,
    #     "label": "Number of Cars",
    #     "min": 1,
    #     "max": 500,
    #     "step": 1,
    # },
    "max_cars":50,
    "p_generate": 0.1,
    "road_points_gdf": road_gdf,
    "max_steps": 30000,
    "log_agents": False,
}

# Create the actual model instance with real values (not Slider dicts)
model = TrafficModel(road_points_gdf=road_gdf, p_generate=0.1, max_cars=10, max_steps=1000, log_agents=False)


# the visulation 
page = SolaraViz(
    TrafficModel,
    components=[make_space_component(agent_portrayal)],
    model_params=model_params,
    name="model",
)
# This is required to render the visualization in the Jupyter notebook
page



# Batch run - not working, try later

In [ ]:
model = TrafficModel(road_points_gdf=road_gdf, p_generate=0.1, max_cars=20, max_mph=60, max_steps=300000, log_agents=False)


perams = {
    "road_points_gdf": road_gdf,
    'max_mph':60, 
    'max_steps':300000,
    'log_agents':False,
    "p_generate": [0.1, 0.001, 0.00001],
    "max_cars": [10, 50 ]
}




results = batch_run(
    model_cls=TrafficModel,
    parameters=perams,
    iterations=1,  # Run each config 5 times
    max_steps=10000,
    data_collection_period=-1,  # Collect at every step, -1 means collect all
    number_processes=1,
    display_progress=True
)

